# Stage 3.1: PagedAttention Explained - Virtual Memory for KV Cache

## Problem Statement
KV cache memory grows quadratically with batch size and is highly fragmented. Traditional allocation wastes 60-80% of memory. How can we achieve near-zero waste and enable efficient prefix sharing?

## What You'll Learn
- PagedAttention concept (vLLM's key innovation)
- Virtual memory analogy for KV cache
- Page-based memory management
- How memory fragmentation is eliminated
- Prefix sharing mechanism
- Real benefits: 20-40% → 60-80% memory utilization

## Prerequisites
- Understanding of KV cache (notebook 10)
- Familiarity with attention mechanism
- Basic OS concepts (virtual memory) - helpful but not required

---

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

## Part 1: The Traditional KV Cache Problem

### Memory Waste in Traditional Approach

Traditional KV cache allocation:
```python
# Pre-allocate contiguous memory for worst case
kv_cache = torch.zeros(
    batch_size,
    num_layers,
    max_seq_length,  # Must allocate for maximum!
    hidden_dim
)
```

**Problems:**
1. **Internal fragmentation**: Must allocate for max length even if actual is shorter
2. **External fragmentation**: Cannot share memory between requests
3. **Static allocation**: Cannot grow dynamically
4. **No sharing**: Cannot reuse common prefixes (system prompts, etc.)

### Visual Example

```
Traditional Allocation:
Request A (50 tokens, allocated 2048):  [################........................] 2.4% used
Request B (200 tokens, allocated 2048): [################........................] 9.8% used
Request C (100 tokens, allocated 2048): [################........................] 4.9% used

Overall memory utilization: 20-40% (60-80% wasted!)
```

In [ ]:
def visualize_traditional_allocation():
    """
    Visualize memory waste in traditional KV cache allocation
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Three requests with different actual lengths
    requests = [
        {'id': 'A', 'actual': 50, 'allocated': 2048, 'y': 3},
        {'id': 'B', 'actual': 200, 'allocated': 2048, 'y': 2},
        {'id': 'C', 'actual': 100, 'allocated': 2048, 'y': 1},
    ]
    
    for req in requests:
        # Allocated space (total)
        ax.barh(req['y'], req['allocated'], height=0.6, 
               color='lightgray', alpha=0.6, edgecolor='black')
        
        # Actually used space
        ax.barh(req['y'], req['actual'], height=0.6,
               color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=2)
        
        # Labels
        utilization = (req['actual'] / req['allocated']) * 100
        ax.text(req['allocated'] + 100, req['y'], 
               f"Request {req['id']}: {utilization:.1f}% used",
               va='center', fontsize=11, fontweight='bold')
        
        # Show wasted space
        wasted = req['allocated'] - req['actual']
        ax.text(req['actual'] + wasted/2, req['y'],
               f"{wasted} tokens\nWASTED",
               ha='center', va='center', fontsize=9, 
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    # Calculate overall utilization
    total_allocated = sum(r['allocated'] for r in requests)
    total_used = sum(r['actual'] for r in requests)
    overall_util = (total_used / total_allocated) * 100
    
    ax.set_xlabel('Memory (tokens)', fontsize=12)
    ax.set_ylabel('Request', fontsize=12)
    ax.set_yticks([1, 2, 3])
    ax.set_yticklabels(['C', 'B', 'A'])
    ax.set_title(f'Traditional KV Cache: Contiguous Allocation\n'
                f'Overall Memory Utilization: {overall_util:.1f}%',
                fontsize=14, fontweight='bold')
    ax.set_xlim(0, 2500)
    
    # Legend
    used_patch = mpatches.Patch(color='#e74c3c', label='Used Memory')
    wasted_patch = mpatches.Patch(color='lightgray', label='Wasted Memory')
    ax.legend(handles=[used_patch, wasted_patch], loc='upper right', fontsize=11)
    
    plt.tight_layout()
    plt.savefig('traditional_kv_cache_waste.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nTraditional Allocation Analysis:")
    print(f"  Total allocated: {total_allocated} tokens")
    print(f"  Total used: {total_used} tokens")
    print(f"  Wasted: {total_allocated - total_used} tokens ({100 - overall_util:.1f}%)")
    print(f"  Memory efficiency: {overall_util:.1f}%")

visualize_traditional_allocation()

---

## Part 2: PagedAttention - Virtual Memory for KV Cache

### The Key Insight: Treat KV Cache Like Virtual Memory

**Operating Systems solved this decades ago!**

OS Virtual Memory:
- Logical addresses (what programs see)
- Physical addresses (actual RAM)
- Page table maps logical → physical
- Pages allocated on-demand
- Can be non-contiguous in physical memory

**PagedAttention applies the same concept to KV cache:**

```
Traditional:
Logical:  [Request A: token 0, 1, 2, ..., max_len]
Physical: [Contiguous block of memory]
Problem: Must allocate max_len upfront, even if unused

PagedAttention:
Logical:  [Request A: token 0-15, 16-31, 32-47, ...]
                         ↓        ↓        ↓
Physical: Page table maps to non-contiguous pages
          [Page 5] [Page 12] [Page 3] ...
Benefit: Allocate pages on-demand, can be anywhere in memory
```

### Page Size Selection

Typical page size: **16 tokens**
- Small enough: Low internal fragmentation
- Large enough: Low page table overhead
- Aligns with GPU memory access patterns

In [ ]:
@dataclass
class Page:
    """Represents a physical page in memory"""
    page_id: int
    tokens_used: int = 0
    page_size: int = 16
    ref_count: int = 0  # For sharing pages
    
    def is_full(self) -> bool:
        return self.tokens_used >= self.page_size
    
    def has_space(self) -> bool:
        return self.tokens_used < self.page_size


@dataclass
class LogicalBlock:
    """Represents a logical block (sequence of tokens) for a request"""
    block_id: int
    physical_page: Optional[Page] = None
    

class PageTable:
    """Maps logical blocks to physical pages"""
    def __init__(self):
        self.mapping: Dict[int, int] = {}  # logical_block_id -> physical_page_id
    
    def map(self, logical_block_id: int, physical_page_id: int):
        self.mapping[logical_block_id] = physical_page_id
    
    def get_physical_page(self, logical_block_id: int) -> Optional[int]:
        return self.mapping.get(logical_block_id)


class PagedMemoryManager:
    """
    Memory manager using paged memory allocation.
    Similar to vLLM's implementation.
    """
    
    def __init__(self, total_pages: int, page_size: int = 16):
        self.page_size = page_size
        self.total_pages = total_pages
        
        # Pool of physical pages
        self.physical_pages = [Page(i, page_size=page_size) for i in range(total_pages)]
        self.free_pages = set(range(total_pages))
        
        # Page tables for each request
        self.page_tables: Dict[int, PageTable] = {}  # request_id -> PageTable
    
    def allocate_request(self, request_id: int):
        """Initialize a new request with its page table"""
        self.page_tables[request_id] = PageTable()
    
    def allocate_page(self, request_id: int, logical_block_id: int) -> Optional[int]:
        """Allocate a physical page for a logical block"""
        if not self.free_pages:
            return None
        
        # Get a free page
        page_id = self.free_pages.pop()
        
        # Map logical block to physical page
        page_table = self.page_tables[request_id]
        page_table.map(logical_block_id, page_id)
        
        return page_id
    
    def free_request(self, request_id: int):
        """Free all pages for a request"""
        if request_id not in self.page_tables:
            return
        
        page_table = self.page_tables[request_id]
        for page_id in page_table.mapping.values():
            self.free_pages.add(page_id)
            self.physical_pages[page_id].tokens_used = 0
        
        del self.page_tables[request_id]
    
    def get_utilization(self) -> float:
        """Calculate memory utilization"""
        used_pages = self.total_pages - len(self.free_pages)
        return (used_pages / self.total_pages) * 100
    
    def get_stats(self) -> Dict:
        return {
            'total_pages': self.total_pages,
            'free_pages': len(self.free_pages),
            'used_pages': self.total_pages - len(self.free_pages),
            'utilization': self.get_utilization(),
            'active_requests': len(self.page_tables)
        }


# Demo
print("\n=== PAGED MEMORY MANAGER DEMO ===")

# Create manager with 128 pages of 16 tokens each = 2048 tokens total
manager = PagedMemoryManager(total_pages=128, page_size=16)

# Allocate three requests with different lengths
requests = [
    (0, 50),   # Request 0: 50 tokens = 4 pages (3 full, 1 partial)
    (1, 200),  # Request 1: 200 tokens = 13 pages (12 full, 1 partial)
    (2, 100),  # Request 2: 100 tokens = 7 pages (6 full, 1 partial)
]

for req_id, num_tokens in requests:
    manager.allocate_request(req_id)
    
    # Calculate number of pages needed
    num_pages = (num_tokens + manager.page_size - 1) // manager.page_size
    
    print(f"\nRequest {req_id}: {num_tokens} tokens → {num_pages} pages")
    
    # Allocate pages
    for block_id in range(num_pages):
        page_id = manager.allocate_page(req_id, block_id)
        tokens_in_page = min(manager.page_size, num_tokens - block_id * manager.page_size)
        manager.physical_pages[page_id].tokens_used = tokens_in_page
        print(f"  Block {block_id} → Page {page_id} ({tokens_in_page} tokens)")

stats = manager.get_stats()
print(f"\n=== MEMORY STATISTICS ===")
print(f"Total capacity: {stats['total_pages']} pages ({stats['total_pages'] * 16} tokens)")
print(f"Used pages: {stats['used_pages']}")
print(f"Free pages: {stats['free_pages']}")
print(f"Utilization: {stats['utilization']:.1f}%")
print(f"Active requests: {stats['active_requests']}")

# Calculate actual tokens used
total_tokens_used = sum(req[1] for req in requests)
total_tokens_capacity = stats['total_pages'] * 16
actual_utilization = (total_tokens_used / total_tokens_capacity) * 100
print(f"\nActual token utilization: {actual_utilization:.1f}%")
print(f"Waste due to partial pages: {stats['utilization'] - actual_utilization:.1f}%")

---

## Part 3: Visual Comparison - Traditional vs PagedAttention

In [ ]:
def visualize_paged_vs_traditional():
    """
    Side-by-side comparison of traditional vs paged memory allocation
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Request specifications
    requests = [
        {'id': 'A', 'tokens': 50, 'color': '#e74c3c'},
        {'id': 'B', 'tokens': 200, 'color': '#3498db'},
        {'id': 'C', 'tokens': 100, 'color': '#2ecc71'},
    ]
    
    # Traditional allocation
    max_alloc = 2048
    y_pos = 0
    
    for req in requests:
        # Allocated space
        ax1.barh(y_pos, max_alloc, height=0.8, 
                color='lightgray', alpha=0.4, edgecolor='black')
        # Used space
        ax1.barh(y_pos, req['tokens'], height=0.8,
                color=req['color'], alpha=0.8, edgecolor='black', linewidth=2)
        
        ax1.text(-100, y_pos, f"Req {req['id']}", 
                ha='right', va='center', fontsize=11, fontweight='bold')
        y_pos += 1
    
    ax1.set_title('Traditional: Contiguous Pre-allocation (20-40% utilization)', 
                 fontsize=14, fontweight='bold')
    ax1.set_xlabel('Memory (tokens)', fontsize=12)
    ax1.set_xlim(-200, 2500)
    ax1.set_yticks([])
    ax1.grid(True, alpha=0.3, axis='x')
    
    # PagedAttention allocation
    page_size = 16
    total_pages = 128
    
    # Create a grid of pages
    pages_per_row = 32
    rows = total_pages // pages_per_row
    
    # Allocate pages for each request
    page_allocation = ['free'] * total_pages
    current_page = 0
    
    for req in requests:
        num_pages = (req['tokens'] + page_size - 1) // page_size
        for _ in range(num_pages):
            if current_page < total_pages:
                page_allocation[current_page] = req['id']
                current_page += 1
    
    # Draw the page grid
    for i in range(total_pages):
        row = i // pages_per_row
        col = i % pages_per_row
        
        if page_allocation[i] == 'free':
            color = 'lightgray'
            alpha = 0.3
        else:
            # Find the request color
            req_color = next(r['color'] for r in requests if r['id'] == page_allocation[i])
            color = req_color
            alpha = 0.8
        
        rect = mpatches.Rectangle((col, rows - row - 1), 1, 1, 
                                  facecolor=color, alpha=alpha, 
                                  edgecolor='black', linewidth=0.5)
        ax2.add_patch(rect)
    
    ax2.set_xlim(0, pages_per_row)
    ax2.set_ylim(0, rows)
    ax2.set_aspect('equal')
    ax2.set_title(f'PagedAttention: On-demand Page Allocation (60-80% utilization)\n'
                 f'Each square = 1 page ({page_size} tokens)',
                 fontsize=14, fontweight='bold')
    ax2.set_xticks([])
    ax2.set_yticks([])
    
    # Add legend
    legend_elements = [
        mpatches.Patch(facecolor=req['color'], alpha=0.8, 
                      edgecolor='black', label=f"Request {req['id']} ({req['tokens']} tokens)")
        for req in requests
    ] + [mpatches.Patch(facecolor='lightgray', alpha=0.3, 
                        edgecolor='black', label='Free/Unused')]
    
    ax2.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('paged_vs_traditional.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Calculate statistics
    total_tokens = sum(r['tokens'] for r in requests)
    traditional_allocated = len(requests) * max_alloc
    traditional_util = (total_tokens / traditional_allocated) * 100
    
    pages_used = sum((r['tokens'] + page_size - 1) // page_size for r in requests)
    paged_allocated = total_pages * page_size
    paged_util = (pages_used * page_size / paged_allocated) * 100
    
    print("\n=== COMPARISON ===")
    print(f"\nTraditional Allocation:")
    print(f"  Total allocated: {traditional_allocated} tokens")
    print(f"  Actually used: {total_tokens} tokens")
    print(f"  Utilization: {traditional_util:.1f}%")
    print(f"  Wasted: {traditional_allocated - total_tokens} tokens")
    
    print(f"\nPagedAttention:")
    print(f"  Total capacity: {paged_allocated} tokens")
    print(f"  Pages used: {pages_used}/{total_pages}")
    print(f"  Utilization: {paged_util:.1f}%")
    print(f"  Efficiency gain: {paged_util / traditional_util:.2f}x")

visualize_paged_vs_traditional()

---

## Part 4: Prefix Sharing - The Killer Feature

### The Problem

Many requests share common prefixes:
- System prompts (same for all users)
- Multi-turn conversations (previous messages)
- Few-shot examples (same examples for all requests)

Traditional approach: **Duplicate the shared prefix for each request**

### PagedAttention Solution: Copy-on-Write

Just like OS virtual memory, pages can be **shared** between requests!

```
Request A: [System Prompt] + [User: Hello] + [AI: Hi] + [User: How are you?]
Request B: [System Prompt] + [User: Hello] + [AI: Hi] + [User: What's 2+2?]
Request C: [System Prompt] + [User: Goodbye]

Traditional: 3 copies of system prompt
PagedAttention: 1 shared copy of system prompt!

Memory savings: Can be 10-20x for long system prompts
```

In [ ]:
class SharedPageManager:
    """
    Extended memory manager with prefix sharing (copy-on-write)
    """
    
    def __init__(self, total_pages: int, page_size: int = 16):
        self.page_size = page_size
        self.total_pages = total_pages
        
        # Pool of physical pages with reference counting
        self.physical_pages = [Page(i, page_size=page_size) for i in range(total_pages)]
        self.free_pages = set(range(total_pages))
        
        # Page tables for each request
        self.page_tables: Dict[int, PageTable] = {}
        
        # Hash table for finding shared prefixes
        # content_hash -> page_id
        self.prefix_cache: Dict[str, List[int]] = {}
    
    def allocate_request_with_prefix(self, 
                                     request_id: int, 
                                     prefix_hash: Optional[str] = None) -> int:
        """Allocate request, sharing prefix pages if available"""
        self.page_tables[request_id] = PageTable()
        pages_shared = 0
        
        # If we have a cached prefix, share those pages
        if prefix_hash and prefix_hash in self.prefix_cache:
            shared_page_ids = self.prefix_cache[prefix_hash]
            
            for block_id, page_id in enumerate(shared_page_ids):
                # Map logical block to existing physical page
                self.page_tables[request_id].map(block_id, page_id)
                
                # Increment reference count (copy-on-write)
                self.physical_pages[page_id].ref_count += 1
                pages_shared += 1
        
        return pages_shared
    
    def register_prefix(self, prefix_hash: str, page_ids: List[int]):
        """Register a prefix for sharing"""
        self.prefix_cache[prefix_hash] = page_ids
        
        # Mark pages as shared
        for page_id in page_ids:
            self.physical_pages[page_id].ref_count += 1
    
    def allocate_page(self, request_id: int, logical_block_id: int) -> Optional[int]:
        """Allocate a new page for a request"""
        if not self.free_pages:
            return None
        
        page_id = self.free_pages.pop()
        self.page_tables[request_id].map(logical_block_id, page_id)
        self.physical_pages[page_id].ref_count = 1
        
        return page_id
    
    def free_request(self, request_id: int):
        """Free pages for a request, handling shared pages"""
        if request_id not in self.page_tables:
            return
        
        page_table = self.page_tables[request_id]
        
        for page_id in page_table.mapping.values():
            page = self.physical_pages[page_id]
            page.ref_count -= 1
            
            # Only free if no other requests are using it
            if page.ref_count == 0:
                self.free_pages.add(page_id)
                page.tokens_used = 0
        
        del self.page_tables[request_id]


# Demo: Prefix Sharing
print("\n=== PREFIX SHARING DEMO ===")

manager = SharedPageManager(total_pages=128, page_size=16)

# Simulate system prompt (5 pages = 80 tokens)
system_prompt_hash = "system_prompt_v1"
system_prompt_pages = []

print("\nAllocating system prompt (80 tokens, 5 pages)...")
manager.allocate_request_with_prefix(999, None)  # Temporary request for setup

for i in range(5):
    page_id = manager.allocate_page(999, i)
    manager.physical_pages[page_id].tokens_used = 16
    system_prompt_pages.append(page_id)
    print(f"  Page {page_id} allocated")

manager.register_prefix(system_prompt_hash, system_prompt_pages)
manager.free_request(999)

print(f"\nSystem prompt registered with hash: {system_prompt_hash}")

# Now allocate 3 user requests, all sharing the system prompt
print("\nAllocating 3 user requests (all share system prompt):")

user_requests = [
    (0, 30, "User query 1"),  # 30 additional tokens
    (1, 50, "User query 2"),  # 50 additional tokens
    (2, 20, "User query 3"),  # 20 additional tokens
]

total_pages_allocated = 0
total_pages_shared = 0

for req_id, additional_tokens, desc in user_requests:
    # Share system prompt
    pages_shared = manager.allocate_request_with_prefix(req_id, system_prompt_hash)
    total_pages_shared += pages_shared
    
    # Allocate pages for user-specific content
    num_additional_pages = (additional_tokens + manager.page_size - 1) // manager.page_size
    
    print(f"\nRequest {req_id} ({desc}):")
    print(f"  Shared: {pages_shared} pages (system prompt)")
    print(f"  New: {num_additional_pages} pages ({additional_tokens} tokens)")
    
    for i in range(num_additional_pages):
        page_id = manager.allocate_page(req_id, 5 + i)  # Start after shared pages
        tokens = min(manager.page_size, additional_tokens - i * manager.page_size)
        manager.physical_pages[page_id].tokens_used = tokens
        total_pages_allocated += 1

print("\n=== SHARING STATISTICS ===")

# Without sharing
without_sharing = 5 * 3 + total_pages_allocated  # 3 copies of system prompt
# With sharing
with_sharing = 5 + total_pages_allocated  # 1 shared copy

print(f"\nWithout prefix sharing: {without_sharing} pages")
print(f"With prefix sharing: {with_sharing} pages")
print(f"Pages saved: {without_sharing - with_sharing}")
print(f"Memory reduction: {(1 - with_sharing/without_sharing) * 100:.1f}%")
print(f"\nSharing efficiency: {total_pages_shared} page references, only 5 physical pages")

# Check reference counts
print("\nReference counts for shared pages:")
for page_id in system_prompt_pages:
    ref_count = manager.physical_pages[page_id].ref_count
    print(f"  Page {page_id}: {ref_count} references")

---

## Part 5: Memory Utilization Benchmark

In [ ]:
def benchmark_memory_utilization(num_requests_list: List[int]):
    """
    Benchmark memory utilization for traditional vs paged allocation
    """
    page_size = 16
    max_seq_len = 2048
    
    traditional_utils = []
    paged_utils = []
    paged_with_sharing_utils = []
    
    for num_requests in num_requests_list:
        # Generate random request lengths
        np.random.seed(42)
        request_lengths = np.random.lognormal(4.5, 0.8, num_requests).astype(int)
        request_lengths = np.clip(request_lengths, 50, 500)
        
        total_tokens = sum(request_lengths)
        
        # Traditional: Each request gets max_seq_len
        traditional_allocated = num_requests * max_seq_len
        traditional_util = (total_tokens / traditional_allocated) * 100
        traditional_utils.append(traditional_util)
        
        # Paged: Allocate only needed pages
        total_pages_needed = sum((length + page_size - 1) // page_size for length in request_lengths)
        total_pages_capacity = (num_requests * max_seq_len) // page_size
        paged_util = (total_pages_needed / total_pages_capacity) * 100
        paged_utils.append(paged_util)
        
        # Paged with prefix sharing (assume 80 token shared prefix)
        shared_prefix_pages = 5  # 80 tokens / 16
        total_pages_with_sharing = shared_prefix_pages + sum(
            (max(0, length - 80) + page_size - 1) // page_size 
            for length in request_lengths
        )
        paged_with_sharing_util = (total_pages_with_sharing / total_pages_capacity) * 100
        paged_with_sharing_utils.append(paged_with_sharing_util)
    
    return traditional_utils, paged_utils, paged_with_sharing_utils


# Run benchmark
num_requests_list = [4, 8, 16, 32, 64]
trad_utils, paged_utils, paged_sharing_utils = benchmark_memory_utilization(num_requests_list)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(num_requests_list))
width = 0.25

bars1 = ax.bar(x - width, trad_utils, width, label='Traditional', 
              color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x, paged_utils, width, label='PagedAttention', 
              color='#3498db', alpha=0.8)
bars3 = ax.bar(x + width, paged_sharing_utils, width, 
              label='PagedAttention + Prefix Sharing', 
              color='#2ecc71', alpha=0.8)

ax.set_xlabel('Number of Requests', fontsize=12)
ax.set_ylabel('Memory Utilization (%)', fontsize=12)
ax.set_title('Memory Utilization: Traditional vs PagedAttention', 
            fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(num_requests_list)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.1f}%',
               ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('memory_utilization_benchmark.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== MEMORY UTILIZATION BENCHMARK ===")
for i, num_req in enumerate(num_requests_list):
    print(f"\n{num_req} requests:")
    print(f"  Traditional:           {trad_utils[i]:.1f}%")
    print(f"  PagedAttention:        {paged_utils[i]:.1f}% ({paged_utils[i]/trad_utils[i]:.2f}x)")
    print(f"  Paged + Sharing:       {paged_sharing_utils[i]:.1f}% ({paged_sharing_utils[i]/trad_utils[i]:.2f}x)")

---

## Part 6: Implementation Details

### Page Table Lookup in Attention

```python
def paged_attention(query, key_cache, value_cache, page_table):
    """
    Compute attention with paged KV cache.
    
    Args:
        query: [batch_size, num_heads, head_dim]
        key_cache: [num_pages, page_size, num_heads, head_dim]
        value_cache: [num_pages, page_size, num_heads, head_dim]
        page_table: [batch_size, max_num_pages] - maps logical to physical pages
    """
    batch_size, num_heads, head_dim = query.shape
    
    outputs = []
    
    for i in range(batch_size):
        # Get physical pages for this request
        physical_pages = page_table[i]  # [max_num_pages]
        
        # Gather keys and values from physical pages
        # This is where the magic happens - non-contiguous access!
        keys = key_cache[physical_pages]  # [num_pages, page_size, num_heads, head_dim]
        values = value_cache[physical_pages]
        
        # Reshape to sequence
        keys = keys.reshape(-1, num_heads, head_dim)  # [seq_len, num_heads, head_dim]
        values = values.reshape(-1, num_heads, head_dim)
        
        # Compute attention
        attn_output = compute_attention(query[i], keys, values)
        outputs.append(attn_output)
    
    return torch.stack(outputs)
```

### Memory Layout

```
Traditional Layout:
[Req 0, Page 0][Req 0, Page 1][Req 0, Page 2]...[Req 1, Page 0][Req 1, Page 1]...
   Contiguous per request

Paged Layout:
[Page 0: Req A][Page 1: Req C][Page 2: Req B][Page 3: Req A][Page 4: Req B]...
   Interleaved, allocated on-demand
```

### Copy-on-Write for Prefix Sharing

```python
def write_to_shared_page(page_id, request_id, page_table):
    """
    When writing to a shared page, copy it first (copy-on-write).
    """
    page = physical_pages[page_id]
    
    if page.ref_count > 1:
        # Page is shared, need to copy
        new_page_id = allocate_new_page()
        physical_pages[new_page_id].data = page.data.clone()
        physical_pages[new_page_id].ref_count = 1
        
        # Update page table
        page_table[request_id][block_id] = new_page_id
        
        # Decrease ref count on old page
        page.ref_count -= 1
        
        return new_page_id
    
    return page_id  # Can write directly
```

---

## Part 7: Real-World Performance Impact

### vLLM Benchmarks (from paper)

**Without PagedAttention:**
- Memory utilization: 20-40%
- Batch size limited by fragmentation
- Frequent OOM errors

**With PagedAttention:**
- Memory utilization: 60-80%
- 2-4x larger batch sizes
- Near-zero OOM errors
- **Throughput: 2-4x higher**

**With PagedAttention + Prefix Caching:**
- Multi-turn conversations: 5-10x speedup
- Shared system prompts: Memory saved proportional to prompt length
- Document QA: 3-5x speedup (shared document context)

### Production Use Cases

1. **Chatbots (Multi-turn)**
   - System prompt: 500 tokens
   - Conversation history: 1000 tokens
   - New turn: 50 tokens
   - Prefix sharing saves: 1500 tokens per turn (30x reduction!)

2. **Document QA**
   - Document: 8000 tokens (shared across all questions)
   - Question: 50 tokens
   - Multiple questions → share document pages
   - Memory savings: 160x for 10 questions

3. **Few-shot Learning**
   - Examples: 2000 tokens (same for all requests)
   - Query: 100 tokens
   - Prefix sharing: 20x reduction per request

---

## Part 8: Key Takeaways

### Why PagedAttention is Revolutionary

1. **Near-Zero Memory Waste**
   - Traditional: 60-80% waste (20-40% utilization)
   - PagedAttention: 20-40% waste (60-80% utilization)
   - **3-4x more efficient memory usage**

2. **Enables Continuous Batching**
   - Dynamic request addition/removal requires flexible memory
   - PagedAttention provides the necessary flexibility
   - Together: 10-20x throughput improvement

3. **Prefix Sharing**
   - Share common prefixes across requests
   - 5-10x speedup for multi-turn conversations
   - Massive memory savings for repeated content

4. **No Pre-allocation Needed**
   - Allocate pages on-demand
   - Can handle variable-length sequences efficiently
   - Reduces OOM errors

### The Virtual Memory Analogy

| OS Virtual Memory | PagedAttention |
|-------------------|----------------|
| Logical addresses | Logical token positions |
| Physical pages | Physical memory pages |
| Page table | Page table (same!) |
| On-demand paging | On-demand allocation |
| Copy-on-write | Copy-on-write for shared prefixes |
| Reference counting | Reference counting for shared pages |

### Production Impact

PagedAttention + Continuous Batching = **10-20x throughput improvement**

This is why vLLM became the standard for production LLM serving.

### When PagedAttention Matters Most

1. **Variable-length requests** - Eliminates fragmentation
2. **Multi-turn conversations** - Prefix sharing is huge win
3. **Shared prompts** - System prompts, few-shot examples
4. **High-throughput serving** - Enables larger batch sizes
5. **Memory-constrained environments** - Better utilization

---

## References & Next Steps

### Papers
- **"Efficient Memory Management for Large Language Model Serving with PagedAttention"** (Kwon et al., 2023) - The vLLM paper
- Original OS virtual memory concepts

### Related Notebooks
- **Notebook 10**: KV cache implementation
- **Notebook 30**: Continuous batching (pairs perfectly with PagedAttention)
- **Notebook 33**: Prefix caching in detail
- **Notebook 34**: Multi-GPU serving

### Production Implementations
- **vLLM**: https://github.com/vllm-project/vllm (original implementation)
- **TensorRT-LLM**: Also uses PagedAttention
- **SGLang**: Alternative implementation with prefix caching

### LLM Inference Roadmap
- See `LLM_INFERENCE_ROADMAP.md` for comprehensive guide
- Stage 3: Advanced Serving section

---

## Summary

PagedAttention is vLLM's **key innovation** that revolutionized LLM serving:

1. Treats KV cache like OS virtual memory
2. Eliminates memory fragmentation (60-80% utilization vs 20-40%)
3. Enables prefix sharing (5-10x speedup for conversations)
4. Works seamlessly with continuous batching
5. **Result: 10-20x higher throughput**

**Key insight:** Apply 50-year-old OS concepts to modern LLM serving.

**Next:** Notebook 33 - Prefix Caching (deep dive into shared prefix optimization)